# Mage Iris Classification – Reproducible Jupyter Notebook

**Companion notebook for the `iris_project` Mage pipeline.**  
This notebook mirrors the submitted Mage blocks: `load_iris`, `prepare_data`, `train_model`, `evaluate_model`, `save_results`, and `export_metrics`.

- Dataset: scikit-learn Iris (150 rows, 4 features, 3 classes)
- Split: 80% training / 20% testing, `random_state=42`, stratified by target
- Classifier: `DecisionTreeClassifier(random_state=42)`
- Depth experiment: `max_depth` 1 through 5
- Main pipeline configuration: `max_depth=3`

Run cells from top to bottom. The code writes `iris_predictions.csv` and `iris_metrics.csv` in the notebook's working directory.


## 1. Import libraries and load the Iris dataset

In [1]:
import os
import pandas as pd
import numpy as np

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report
)

iris = load_iris(as_frame=True)
df = iris.frame.copy()
df["target_name"] = df["target"].map(dict(enumerate(iris.target_names)))

print("Dataset shape:", df.shape)
print("Class counts:")
display(df["target_name"].value_counts().rename_axis("class").to_frame("records"))
assert len(df) == 150
assert df["target"].nunique() == 3
assert df.isnull().sum().sum() == 0


Dataset shape: (150, 6)
Class counts:


,records
class,
setosa,50
versicolor,50
virginica,50


## 2. Prepare the same 80:20 split used by Mage

In [2]:
FEATURE_COLUMNS = [
    "sepal length (cm)",
    "sepal width (cm)",
    "petal length (cm)",
    "petal width (cm)",
]

train_idx, test_idx = train_test_split(
    df.index,
    test_size=0.2,
    random_state=42,
    stratify=df["target"],
)

train_df = df.loc[train_idx].copy()
test_df = df.loc[test_idx].copy()

X_train = train_df[FEATURE_COLUMNS]
y_train = train_df["target"]
X_test = test_df[FEATURE_COLUMNS]
y_test = test_df["target"]

print("Training rows:", len(train_df))
print("Testing rows:", len(test_df))
print("Split ratio: 80:20")
assert len(train_df) == 120
assert len(test_df) == 30


Training rows: 120
Testing rows: 30
Split ratio: 80:20


## 3. Experiment: compare `max_depth` values 1–5

This reproduces the Part C terminal experiment. Each model uses the same train/test rows and `random_state=42`.

In [3]:
depth_results = []

for depth in range(1, 6):
    model = DecisionTreeClassifier(max_depth=depth, random_state=42)
    model.fit(X_train, y_train)
    depth_predictions = model.predict(X_test)
    accuracy = accuracy_score(y_test, depth_predictions)
    depth_results.append({
        "max_depth": depth,
        "test_accuracy": accuracy,
        "test_accuracy_percent": round(accuracy * 100, 2),
        "meets_90_percent_threshold": accuracy >= 0.90,
    })

depth_results_df = pd.DataFrame(depth_results)
display(depth_results_df)


,max_depth,test_accuracy,test_accuracy_percent,meets_90_percent_threshold
0,1,0.666667,66.67,False
1,2,0.933333,93.33,True
2,3,0.966667,96.67,True
3,4,0.933333,93.33,True
4,5,0.933333,93.33,True


## 4. Train the final Mage configuration (`max_depth=3`)

In [4]:
MAX_DEPTH = 3

model = DecisionTreeClassifier(max_depth=MAX_DEPTH, random_state=42)
model.fit(X_train, y_train)

predictions = model.predict(X_test)
final_accuracy = accuracy_score(y_test, predictions)

prediction_results = pd.DataFrame({
    "actual": y_test.values,
    "predicted": predictions,
    "max_depth": MAX_DEPTH,
    "test_accuracy": final_accuracy,
})

print(f"Max depth: {MAX_DEPTH}")
print(f"Test accuracy: {final_accuracy:.4f} ({final_accuracy * 100:.2f}%)")
print("Prediction rows:", len(prediction_results))
display(prediction_results.head(10))

assert final_accuracy >= 0.90
assert len(prediction_results) == 30


Max depth: 3
Test accuracy: 0.9667 (96.67%)
Prediction rows: 30


,actual,predicted,max_depth,test_accuracy
0,0,0,3,0.966667
1,2,2,3,0.966667
2,1,1,3,0.966667
3,1,1,3,0.966667
4,0,0,3,0.966667
5,1,1,3,0.966667
6,0,0,3,0.966667
7,0,0,3,0.966667
8,2,2,3,0.966667
9,1,1,3,0.966667


## 5. Evaluate: confusion matrix and classification metrics

In [5]:
cm = confusion_matrix(y_test, predictions, labels=[0, 1, 2])
print("CONFUSION MATRIX")
print("Rows = Actual; Columns = Predicted")
display(pd.DataFrame(
    cm,
    index=[f"Actual {name}" for name in iris.target_names],
    columns=[f"Predicted {name}" for name in iris.target_names],
))

report = classification_report(
    y_test,
    predictions,
    labels=[0, 1, 2],
    target_names=["setosa", "versicolor", "virginica"],
    output_dict=True,
    zero_division=0,
)
metrics_df = pd.DataFrame(report).transpose().reset_index(names="class")
print("CLASSIFICATION METRICS")
display(metrics_df.round(4))


CONFUSION MATRIX
Rows = Actual; Columns = Predicted


,Predicted setosa,Predicted versicolor,Predicted virginica
Actual setosa,10,0,0
Actual versicolor,0,9,1
Actual virginica,0,0,10


CLASSIFICATION METRICS


,class,precision,recall,f1-score,support
0,setosa,1.0000,1.0000,1.0000,10.0000
1,versicolor,1.0000,0.9000,0.9474,10.0000
2,virginica,0.9091,1.0000,0.9524,10.0000
3,accuracy,0.9667,0.9667,0.9667,0.9667
4,macro avg,0.9697,0.9667,0.9666,30.0000
5,weighted avg,0.9697,0.9667,0.9666,30.0000


## 6. Export predictions and metrics (same outputs as Mage exporters)

In [6]:
predictions_path = os.path.abspath("iris_predictions.csv")
metrics_path = os.path.abspath("iris_metrics.csv")

prediction_results.to_csv(predictions_path, index=False)
metrics_df.to_csv(metrics_path, index=False)

print("Predictions saved to:", predictions_path)
print("Total predictions:", len(pd.read_csv(predictions_path)))
print("Metrics saved to:", metrics_path)
print("Total metric rows:", len(pd.read_csv(metrics_path)))

saved_predictions = pd.read_csv(predictions_path)
saved_metrics = pd.read_csv(metrics_path)

assert len(saved_predictions) == 30
assert {"actual", "predicted", "test_accuracy"}.issubset(saved_predictions.columns)
assert len(saved_metrics) >= 5
assert {"class", "precision", "recall", "f1-score"}.issubset(saved_metrics.columns)
print("Exporter validation: PASSED")


Predictions saved to: /home/oai/iris_predictions.csv
Total predictions: 30
Metrics saved to: /home/oai/iris_metrics.csv
Total metric rows: 6
Exporter validation: PASSED


## 7. Part C validation summary

In [7]:
validation_summary = pd.DataFrame([
    ["Dataset contains 150 records", len(df) == 150],
    ["80:20 split gives 120 train / 30 test", len(train_df) == 120 and len(test_df) == 30],
    ["Depth 3 accuracy is at least 90%", final_accuracy >= 0.90],
    ["Prediction CSV contains 30 rows", len(saved_predictions) == 30],
    ["Metrics CSV contains required columns", {"class", "precision", "recall", "f1-score"}.issubset(saved_metrics.columns)],
], columns=["validation", "passed"])

display(validation_summary)
assert validation_summary["passed"].all()
print("ALL PART C NOTEBOOK CHECKS PASSED")


,validation,passed
0,Dataset contains 150 records,True
1,80:20 split gives 120 train / 30 test,True
2,Depth 3 accuracy is at least 90%,True
3,Prediction CSV contains 30 rows,True
4,Metrics CSV contains required columns,True


ALL PART C NOTEBOOK CHECKS PASSED
